# **Noise2Noise (2D) — powered by CAREamics**

---

<font size = 4> Noise2Noise is a deep-learning method that denoises images by training on **pairs of independently-noisy images of the same scene** — no clean, high-quality ground truth is required. It was originally published by [Lehtinen *et al.* (2018)](https://arxiv.org/abs/1803.04189). The network learns to map one noisy image to a second, independent noisy acquisition of the same content; because the two noise realisations are independent, the network converges to the underlying clean signal.

<font size = 4> **This notebook runs Noise2Noise on 2D datasets using [CAREamics](https://careamics.github.io/), a modern PyTorch/Lightning implementation.**

---

<font size = 4>*Disclaimer*:

<font size = 4>This notebook is part of the Zero-Cost Deep-Learning to Enhance Microscopy project (https://github.com/HenriquesLab/DeepLearning_Collab/wiki). Jointly developed by the Jacquemet (https://cellmig.org/) and Henriques (https://henriqueslab.github.io/) laboratories.

<font size = 4>The deep-learning engine used here is **CAREamics** (https://github.com/CAREamics/careamics).

<font size = 4>This notebook is based on:

<font size = 4>**Noise2Noise: Learning Image Restoration without Clean Data**, Lehtinen *et al.*, ICML 2018 (https://arxiv.org/abs/1803.04189)

<font size = 4>**Please cite the original Noise2Noise paper and CAREamics when using this notebook.**

# **How to use this notebook?**

---

<font size = 4>This notebook is structured in numbered sections. Run the cells from top to bottom.

---
### **Structure of a notebook**

<font size = 4>**Text cells** provide information. **Code cells** contain code; move your cursor over the `[ ]` on the left and click the play button to execute.

---
### **Making changes to the notebook**

<font size = 4>**Make a copy** of this notebook and save it to your Google Drive (`File -> Save a copy in Drive`) before editing.

# **0. Before getting started**
---

<font size = 4>Before running the notebook, make sure you are logged into your Google account and that your data is in your Google Drive.

<font size = 4>Noise2Noise requires **paired training data**: for each field of view you need **two independently-acquired noisy images** of the same scene. One acts as the input, the other as the target.

<font size = 4>Please note that you can **only use .tif files!**

<font size = 4>The input and target images must be provided in **two separate folders**, and the paired images must have **matching file names** (they are paired in sorted order).

<font size = 4>A common data structure that works well:

*   Data
    - **Training**
        - source (noisy) — img_1.tif, img_2.tif ...
        - target (independently noisy, same scenes) — img_1.tif, img_2.tif ...
    - **Quality control** (optional but recommended)
        - Low SNR images — img_1.tif, img_2.tif ...
        - High SNR images — img_1.tif, img_2.tif ...
    - **Prediction** — images to denoise
    - **Results**

---
<font size = 4>**Important note**

<font size = 4>- To **train from scratch**: run **sections 1–4**, then **section 5** to assess quality and **section 6** to predict.
<font size = 4>- To only **run predictions** with an existing model: run **sections 1–2**, then **section 6**.
---

# **1. Install CAREamics and dependencies**
---

## **1.1. Install CAREamics**

In [ ]:
#@markdown ##Install CAREamics and dependencies
#@markdown This installs a pinned, tested version of CAREamics. It may take a minute.

# NumPy is pinned to <2.1 to match the version Colab already ships (and which
# CAREamics 0.3.2 supports: numpy>=1.21,<=2.4.6). This stops pip from upgrading
# NumPy inside the running kernel, which would otherwise break imports (and
# Colab's pre-installed numba) and force a runtime restart.
!pip install "careamics==0.3.2" "careamics-portfolio" "numpy<2.1" -q

print("CAREamics installed.")

## **1.2. Restart the runtime (only if you see an import error)**
<font size = 4>The install above keeps Colab's existing NumPy, so you can normally continue straight to section 1.3. **If section 1.3 raises a NumPy or import error**, go to `Runtime -> Restart session`, then re-run from section 1.3 (do **not** re-run the install cell).

## **1.3. Load key dependencies**

In [ ]:
#@markdown ##Load key dependencies
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

import careamics
from careamics import CAREamist
from careamics.config import create_n2n_config
from careamics.metrics.metrics import psnr, scale_invariant_psnr

print(f"CAREamics version: {careamics.__version__}")

# **2. Initialise the Colab session**
---

## **2.1. Check for GPU access**

In [ ]:
#@markdown ##Run this cell to check if you have GPU access
import torch

if torch.cuda.is_available():
    print("You have GPU access.")
    print(torch.cuda.get_device_name(0))
else:
    print("You do NOT have GPU access.")
    print("Go to 'Runtime -> Change runtime type' and select a GPU hardware accelerator,")
    print("then re-run the notebook. Expect slow performance on CPU.")

## **2.2. Mount your Google Drive**

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab
#@markdown * Follow the instructions.
#@markdown * Click on "Files" on the left. Refresh it — your Google Drive appears as "drive".

from google.colab import drive
drive.mount('/content/gdrive')

## **2.3. (Optional) Download an example dataset**
<font size = 4>If you just want to try the notebook without your own data, run this cell to download the **SEM** Noise2Noise example dataset. It contains several independently-noisy acquisitions of the same scene; two of them are written to paired `input/` and `target/` folders. The printed paths can be pasted into `Training_source` and `Training_target` in section 3.1.

In [ ]:
#@markdown ##(Optional) Download the SEM example dataset for testing
Download_example_dataset = True #@param {type:"boolean"}

if Download_example_dataset:
    from careamics_portfolio import PortfolioManager

    portfolio = PortfolioManager()
    download = portfolio.denoising.N2N_SEM.download("./example_data_n2n")
    tif_files = sorted(f for f in download if str(f).endswith("tif"))
    # The SEM stack holds 7 acquisitions of the same scene at different scan times.
    stack = tifffile.imread(tif_files[1])

    in_dir = Path("example_data_n2n/input")
    tgt_dir = Path("example_data_n2n/target")
    in_dir.mkdir(parents=True, exist_ok=True)
    tgt_dir.mkdir(parents=True, exist_ok=True)
    # Two independent 1 us-scan acquisitions of the same scene form a Noise2Noise pair.
    tifffile.imwrite(in_dir / "scene_00.tif", stack[2])
    tifffile.imwrite(tgt_dir / "scene_00.tif", stack[3])

    print("Training_source (input): ", in_dir)
    print("Training_target (target):", tgt_dir)
    print("Paste these into Training_source and Training_target in section 3.1.")

# **3. Select your parameters and paths**
---

## **3.1. Setting the main training parameters**
<font size = 4>`Training_source` and `Training_target` should each point to a folder of `.tif` images. The two folders must contain the **same number of images with matching file names** (paired in sorted order).

In [ ]:
#@markdown ###Path to the noisy input images (folder of .tif files):
Training_source = "" #@param {type:"string"}
#@markdown ###Path to the independently-noisy target images (folder of .tif files):
Training_target = "" #@param {type:"string"}

#@markdown ### Model name and output folder:
model_name = "my_n2n_model" #@param {type:"string"}
model_path = "" #@param {type:"string"}

#@markdown ###Training parameters
#@markdown Number of epochs:
number_of_epochs = 100 #@param {type:"number"}
#@markdown Patch size (pixels, square):
patch_size = 64 #@param {type:"number"}
#@markdown Batch size:
batch_size = 64 #@param {type:"number"}
#@markdown Number of patches held out for validation:
n_val_patches = 8 #@param {type:"number"}

## **3.2. Data augmentation**
<font size = 4>Data augmentation (flips and 90° rotations) usually improves results and is recommended.

In [ ]:
#@markdown ##Enable or disable data augmentation:
Use_Data_augmentation = True #@param {type:"boolean"}

## **3.3. Using a pre-trained model**
<font size = 4>You can continue training from a previously trained CAREamics model. Provide the path to a checkpoint (`.ckpt`). The pre-trained model's configuration is reused, so the parameters above are ignored when this is enabled.

In [ ]:
#@markdown ##Load weights from a pre-trained CAREamics model
Use_pretrained_model = False #@param {type:"boolean"}
#@markdown ###If enabled, provide the path to the checkpoint (.ckpt) file:
pretrained_model_path = "" #@param {type:"string"}

# **4. Train the network**
---

## **4.1. Prepare the training data and model**

In [ ]:
#@markdown ##Create the configuration and the CAREamist
# Augmentations: None -> default (flips + 90-degree rotations); [] -> disabled
augmentations = None if Use_Data_augmentation else []

work_dir = Path(model_path) if model_path else Path(".")
work_dir.mkdir(parents=True, exist_ok=True)

if Use_pretrained_model:
    print(f"Loading pre-trained model from: {pretrained_model_path}")
    careamist = CAREamist(checkpoint_path=pretrained_model_path, work_dir=work_dir)
else:
    config = create_n2n_config(
        experiment_name=model_name,
        data_type="tiff",
        axes="YX",
        patch_size=(patch_size, patch_size),
        batch_size=batch_size,
        num_epochs=number_of_epochs,
        n_val_patches=n_val_patches,
        augmentations=augmentations,
    )
    print(config)
    careamist = CAREamist(config, work_dir=work_dir)

## **4.2. Start training**
<font size = 4>Training checkpoints are saved automatically to your output folder. You can monitor the loss below and in Section 5.

In [ ]:
#@markdown ##Start training
careamist.train(
    train_data=Path(Training_source),
    train_data_target=Path(Training_target),
)
print("Training complete.")

# **5. Evaluate your model**
---

## **5.1. Inspection of the loss function**

In [ ]:
#@markdown ##Plot the training and validation loss vs. epoch
loss_dict = careamist.get_losses()
plt.figure(figsize=(8, 5))
plt.plot(loss_dict["train_epoch"], loss_dict["train_loss"], label="Train loss")
plt.plot(loss_dict["val_epoch"], loss_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training losses")
plt.show()

## **5.2. Quality metrics estimation**
<font size = 4>If you have a Quality Control dataset (paired low-SNR and high-SNR images), the notebook denoises the low-SNR images and compares them against the high-SNR targets using PSNR and scale-invariant PSNR.

In [ ]:
#@markdown ##Provide the Quality Control folders (paired low-SNR and high-SNR .tif images)
Source_QC_folder = "" #@param {type:"string"}
Target_QC_folder = "" #@param {type:"string"}

source_files = sorted(Path(Source_QC_folder).glob("*.tif"))
target_files = sorted(Path(Target_QC_folder).glob("*.tif"))

psnrs, si_psnrs = [], []
for src_f, tgt_f in zip(source_files, target_files):
    gt = tifffile.imread(tgt_f).astype(np.float32)
    pred, _ = careamist.predict(pred_data=str(src_f), tile_size=(256, 256))
    pred_img = np.asarray(pred[0]).squeeze()
    data_range = gt.max() - gt.min()
    psnrs.append(psnr(gt, pred_img, data_range=data_range))
    si_psnrs.append(scale_invariant_psnr(gt, pred_img))
    print(f"{src_f.name}: PSNR={psnrs[-1]:.2f}, SI-PSNR={si_psnrs[-1]:.2f}")

if psnrs:
    print(f"\nMean PSNR:    {np.mean(psnrs):.2f} +/- {np.std(psnrs):.2f}")
    print(f"Mean SI-PSNR: {np.mean(si_psnrs):.2f} +/- {np.std(si_psnrs):.2f}")

# **6. Using the trained model**
---

## **6.1. Generate predictions from an unseen dataset**

In [ ]:
#@markdown ###Path to the data to denoise and the folder where results are saved:
Data_folder = "" #@param {type:"string"}
Result_folder = "" #@param {type:"string"}

result_dir = Path(Result_folder)
result_dir.mkdir(parents=True, exist_ok=True)

predictions, sources = careamist.predict(
    pred_data=Path(Data_folder),
    tile_size=(256, 256),
)

for pred, source in zip(predictions, sources):
    out_name = Path(source).stem + "_denoised.tif"
    out_path = result_dir / out_name
    tifffile.imwrite(out_path, np.asarray(pred).squeeze().astype(np.float32))
    print(f"Saved: {out_path}")

## **6.2. Assess the predicted output**

In [ ]:
#@markdown ##Display an input image next to its denoised prediction
idx = 0 #@param {type:"number"}

input_files = sorted(Path(Data_folder).glob("*.tif"))
input_img = tifffile.imread(input_files[idx])
pred_img = np.asarray(predictions[idx]).squeeze()

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(input_img, cmap="gray")
ax[0].set_title("Input (noisy)")
ax[1].imshow(pred_img, cmap="gray")
ax[1].set_title("Prediction (denoised)")
for a in ax:
    a.axis("off")
plt.show()

# **7. Version log**
---
<font size = 4>**v1.0 (CAREamics)**:
*   First release of the CAREamics-powered Noise2Noise 2D notebook.
*   Uses CAREamics 0.3.2 (`create_n2n_config` + `CAREamist`).

# **Thank you for using Noise2Noise 2D!**